# 🎬 StudioOps - Media Worker (Google Colab T4 GPU)

Este notebook executa o **Músculo de Mídia** do StudioOps:
1. **Text-to-Speech:** Edge-TTS gerando narração neural em PT-BR (0 VRAM);
2. **Geração de Imagens:** SDXL Turbo em FP16 (768x1344 vertical 9:16, <8GB VRAM);
3. **Montagem de Vídeo:** MoviePy com efeito Ken Burns (movimento contínuo de câmera 9:16);
4. **API FastAPI + Ngrok:** Expõe o endpoint público HTTPS para o **n8n** consumir.

---

### 1. Verificar GPU T4
Certifique-se de que o ambiente está com **GPU T4** ativado em *Ambiente de Execução > Alterar tipo de ambiente de execução > GPU T4*.

In [ ]:
!nvidia-smi

### 2. Instalar Dependências

In [ ]:
!pip install -q fastapi uvicorn pyngrok diffusers transformers accelerate edge-tts moviepy pillow imageio nest_asyncio

### 3. Configurar Ngrok e Inicializar Servidor FastAPI

In [ ]:
import os
from pyngrok import ngrok

# Cole seu token do Ngrok entre as aspas abaixo:
NGROK_TOKEN = "[SEU-TOKEN-AQUI]"

# Configura o token no ambiente
!ngrok config add-authtoken $NGROK_TOKEN

# Fecha conexões antigas e abre o novo túnel na porta 8000
ngrok.kill()
http_tunnel = ngrok.connect(addr="http://127.0.0.1:8000")
public_url = http_tunnel.public_url

print('=' * 65)
print('🚀 COPIE ESTA URL PARA O N8N:')
print(public_url)
print('=' * 65)

### 4. Código do Servidor de Mídia e Modelos de IA

In [ ]:
import torch
import asyncio
import nest_asyncio
import threading
import uvicorn
from pathlib import Path
from fastapi import FastAPI, BackgroundTasks, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
from typing import List, Optional, Dict, Any
from diffusers import AutoPipelineForText2Image
import edge_tts
from moviepy.editor import ImageClip, AudioFileClip, concatenate_videoclips

nest_asyncio.apply()

# Diretórios
OUT_DIR = Path('output')
IMG_DIR = OUT_DIR / 'images'
VID_DIR = OUT_DIR / 'videos'
AUD_DIR = OUT_DIR / 'audio'
for d in [IMG_DIR, VID_DIR, AUD_DIR]:
    d.mkdir(parents=True, exist_ok=True)

import gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# Carregamento otimizado do SDXL Turbo (FP16 com CPU Offload)
print('Carregando SDXL Turbo em FP16...')
pipe = AutoPipelineForText2Image.from_pretrained(
    'stabilityai/sdxl-turbo',
    torch_dtype=torch.float16,
    variant='fp16'
)
if torch.cuda.is_available():
    pipe.enable_model_cpu_offload()
print('SDXL carregado com sucesso!')

app = FastAPI(title='StudioOps Colab Worker')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

JOBS: Dict[str, Dict[str, Any]] = {}

class SceneInput(BaseModel):
    order: int
    timeRange: Optional[str] = '0-3s'
    narration: Optional[str] = ''
    prompt: Optional[str] = 'Cinematic scene, vertical 9:16'

class RenderRequest(BaseModel):
    jobId: Optional[str] = None
    topic: str
    style: str = 'Cyberpunk'
    voice: str = 'pt-BR-FranciscaNeural'
    narration: str = ''
    scenes: List[SceneInput]
    callbackWebhook: Optional[str] = None

async def process_job(job_id: str, req: RenderRequest, base_url: str):
    try:
        # 1. TTS
        JOBS[job_id]['status'] = 'generating_audio'
        audio_file = AUD_DIR / f'{job_id}.mp3'
        text = req.narration or ' '.join([s.narration for s in req.scenes])
        comm = edge_tts.Communicate(text, req.voice)
        await comm.save(str(audio_file))

        # 2. Imagens (SDXL 9:16)
        JOBS[job_id]['status'] = 'generating_images'
        img_paths = []
        for sc in req.scenes:
            p = sc.prompt or f'{req.topic}, {req.style}, vertical 9:16, 8k'
            out_img = IMG_DIR / f'{job_id}_scene_{sc.order}.png'
            img = pipe(prompt=p, width=576, height=1024, num_inference_steps=4, guidance_scale=0.0).images[0]
            img.save(out_img)
            img_paths.append(out_img)

        # 3. Montagem com Ken Burns (Cálculo dinâmico para 15s)
        JOBS[job_id]['status'] = 'rendering_video'
        if len(img_paths) == 4:
            durations = [3.0, 4.0, 4.0, 4.0]
        elif len(img_paths) == 3:
            durations = [3.0, 8.0, 4.0]
        else:
            durations = [15.0 / max(1, len(img_paths))] * len(img_paths)
        tot_dur = sum(durations)
        clips = []
        for i, pth in enumerate(img_paths):
            cl = ImageClip(str(pth)).set_duration(durations[i]).resize(height=1920)
            if cl.w < 1080:
                cl = cl.resize(width=1080)
            # Zoom suave contínuo
            cl = cl.resize(lambda t: 1.0 + 0.03 * t)
            clips.append(cl)

        final_vid = concatenate_videoclips(clips, method='compose').set_duration(tot_dur)
        if audio_file.exists():
            try:
                aud = AudioFileClip(str(audio_file))
                if aud.duration > tot_dur:
                    aud = aud.subclip(0, tot_dur)
                final_vid = final_vid.set_audio(aud)
            except Exception as aud_err:
                print(f'Aviso no audio: {aud_err}')

        out_vid = VID_DIR / f'{job_id}.mp4'
        final_vid.write_videofile(str(out_vid), fps=30, codec='libx264', audio_codec='aac', preset='fast', logger=None)

        JOBS[job_id]['status'] = 'completed'
        JOBS[job_id]['progress'] = 100
        target_base = public_url if 'public_url' in globals() and public_url else base_url
        if '127.0.0.1' in target_base or 'localhost' in target_base:
            if 'public_url' in globals() and public_url:
                target_base = public_url
        JOBS[job_id]['videoUrl'] = f'{target_base}/video/{job_id}.mp4'
    except Exception as e:
        JOBS[job_id]['status'] = 'error'
        JOBS[job_id]['error'] = str(e)

@app.get('/health')
def health():
    return {'status': 'ok', 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}

@app.post('/render')
async def render(req: RenderRequest, bg: BackgroundTasks, r: Request):
    jid = req.jobId or f'job_{len(JOBS)+1}'
    base = str(r.base_url).rstrip('/')
    JOBS[jid] = {'status': 'queued', 'progress': 5, 'videoUrl': None}
    bg.add_task(process_job, jid, req, base)
    return {'success': True, 'job_id': jid, 'status_url': f'{base}/jobs/{jid}'}

@app.get('/jobs/{jid}')
def get_job(jid: str):
    return JOBS.get(jid, {'error': 'not found'})

@app.get('/video/{fname}')
def get_video(fname: str):
    f = VID_DIR / fname
    return FileResponse(str(f), media_type='video/mp4')

# Executa uvicorn em thread secundária
def run_server():
    config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='info')
    server = uvicorn.Server(config)
    server.run()

t = threading.Thread(target=run_server, daemon=True)
t.start()
import time, urllib.request
time.sleep(2)
try:
    with urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=3) as r:
        print('✅ Servidor FastAPI rodando e respondendo em http://127.0.0.1:8000!')
except Exception as e:
    print('Aviso:', e)


### 5. Teste Rápido de Renderização Local no Colab

In [ ]:
import requests
import time
from IPython.display import HTML

test_payload = {
    'topic': 'Como a IA cria vídeos em segundos',
    'style': 'Cyberpunk',
    'voice': 'pt-BR-FranciscaNeural',
    'narration': 'Você ainda perde quatro horas editando um vídeo? O StudioOps orquestra áudio e vídeo na nuvem. Comente TESTAR!',
    'scenes': [
        {'order': 1, 'timeRange': '0-3s', 'narration': 'Você ainda perde quatro horas editando um vídeo?', 'prompt': 'Close-up glowing human eye reflecting neon code, cyberpunk, 8k, vertical 9:16'},
        {'order': 2, 'timeRange': '3-12s', 'narration': 'O StudioOps orquestra áudio e vídeo na nuvem.', 'prompt': 'Futuristic glass desk with 3d holographic timeline, neon volumetric light, 8k, 9:16'},
        {'order': 3, 'timeRange': '12-15s', 'narration': 'Comente TESTAR para receber agora!', 'prompt': 'Silhouette of creator pointing to camera with glowing neon city background, 9:16'}
    ]
}

API_URL = public_url
r = requests.post(f'{API_URL}/render', json=test_payload)
print('Status HTTP:', r.status_code)
res = r.json()
print('Job submetido:', res)

jid = res['job_id']
while True:
    st = requests.get(f'{API_URL}/jobs/{jid}').json()
    print('Status:', st.get('status'))
    if st.get('status') == 'completed':
        print('🎉 VÍDEO CONCLUÍDO:', st.get('videoUrl'))
        display(HTML(f'<video width="320" height="568" controls><source src="{st.get("videoUrl")} " type="video/mp4"></video>'))
        break
    elif st.get('status') == 'error':
        print('Erro:', st)
        break
    time.sleep(5)